[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/computational-chemical-biology/AdvancesMSDataProcessing/blob/master/PredictiveModelingBioactivityChemicalProperties/FERMO_core_case_study.ipynb)


> **Source:** [github.com](https://github.com/fermo-metabolomics)


# Running `fermo_core` — the FERMO Planomonospora case study

`fermo_core` is the analysis engine behind the [FERMO](https://github.com/fermo-metabolomics/FERMO) metabolomics
dashboard. The dashboard itself (the repo you linked) is a Flask + Celery + Redis web app with no notebook-friendly
API — it exists to run `fermo_core` as a background job and show the results in a browser. `fermo_core`,
by contrast, is a plain installable library (`pip install fermo_core`) with a small, well-defined entry point,
so this notebook drives it directly, the same way [`fermo_core/main.py`](https://github.com/fermo-metabolomics/fermo_core/blob/main/fermo_core/main.py)
does internally — just without the CLI argument parsing and file-based log handler.

**The dataset.** We reproduce the demo shipped with `fermo_core`: LC-MS/MS extracts of bacteria from the genus
*Planomonospora*, screened for antibiotic activity against *Staphylococcus aureus*
([Helfrich et al. 2021](https://doi.org/10.1021/acs.jnatprod.0c00807)). Running the pipeline and filtering
by `Phenotype Score` recovers a cluster of features annotated as **siomycins**, thiopeptide antibiotics —
feature ID `83` is the one called out in the FERMO README.

**Pipeline stages** (mirroring `main()` in `fermo_core/main.py`):

1. `ParameterManager` — load and validate the JSON parameter file against `fermo_core`'s JSON Schema
2. `GeneralParser` — parse the peak table, MS/MS spectra, phenotype data, group metadata, spectral library,
   MS2Query results and antiSMASH results into `Stats` / feature and sample `Repository` objects
3. `AnalysisManager` — run every enabled analysis module (filtering, blank/group assignment, phenotype
   scoring, spectral similarity networking, annotation, novelty scoring, chromatogram trace calculation)
4. `ExportManager` — write `out.fermo.session.json`, `out.fermo.full.csv` / `out.fermo.abbrev.csv`,
   Cytoscape-compatible `.graphml` networks, and a human-readable summary

Everything downstream of step 4 (the interactive dashboard) lives in the `FERMO` / `fermo_gui` repo and is
out of scope for a notebook — `out.fermo.session.json` is exactly the file you'd drop into
[FERMO Online](https://fermo.bioinformatics.nl/) to browse the results in the GUI.

## 1. Setup

`fermo_core` requires **Python 3.11** (not 3.12+) and is Linux-only for local installation. If you're running
this notebook somewhere else, use a 3.11 virtual environment or Docker.

In [ ]:
pip install fermo_core

In [ ]:
import logging
import platform
import sys
from datetime import datetime
from importlib import metadata
from pathlib import Path

import coloredlogs

from fermo_core.data_analysis.class_analysis_manager import AnalysisManager
from fermo_core.data_processing.parser.class_general_parser import GeneralParser
from fermo_core.input_output.class_export_manager import ExportManager
from fermo_core.input_output.class_file_manager import FileManager
from fermo_core.input_output.class_parameter_manager import ParameterManager
from fermo_core.input_output.class_validation_manager import ValidationManager

print(f"fermo_core version: {metadata.version('fermo_core')}")
print(f"Python version: {platform.python_version()}")

A plain `logging` setup is enough for a notebook — `main_cli()` additionally writes a rotating file
log next to the results, which we skip here since we already get everything printed inline.

In [ ]:
logger = logging.getLogger("fermo_core")
logger.setLevel(logging.DEBUG)

console_handler = logging.StreamHandler(sys.stdout)
console_handler.setLevel(logging.INFO)  # use logging.DEBUG for much more verbose output
console_handler.setFormatter(
    coloredlogs.ColoredFormatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
)
logger.addHandler(console_handler)

## 2. Get the example data

`fermo_core` ships a small `example_data/` folder in its own repository with everything the case study needs:
the MZmine3 peak table, an `.mgf` of MS/MS spectra, qualitative bioactivity data, sample group metadata, a
small spectral library, precomputed MS2Query annotation results, and an antiSMASH results directory for the
*Planomonospora* genome. We clone just that folder with a sparse checkout.

In [ ]:
!git clone --depth 1 --filter=blob:none --sparse https://github.com/fermo-metabolomics/fermo_core.git fermo_core_repo
!cd fermo_core_repo && git sparse-checkout set example_data
!ls fermo_core_repo/example_data

In [ ]:
import shutil

DATA_DIR = Path("example_data")
if not DATA_DIR.exists():
    shutil.copytree("fermo_core_repo/example_data", DATA_DIR)

sorted(p.name for p in DATA_DIR.iterdir())

## 3. Inspect the parameters file

`fermo_core` takes **all** input file paths and module settings from a single JSON file, validated against
[`fermo_core/config/schema.json`](https://github.com/fermo-metabolomics/fermo_core/blob/main/fermo_core/config/schema.json).
This is the case study's parameter file as shipped, unmodified — every `PeaktableParameters` /
`MsmsParameters` / ... block corresponds to one input file or one analysis module, and a module is skipped
entirely if its block is absent or `"activate_module": false`.

In [ ]:
import json

with open("example_data/case_study_parameters.json") as f:
    case_study_params = json.load(f)

print(json.dumps(case_study_params, indent=2))

## 4. Stage 1 — load and validate parameters (`ParameterManager`)

This is the first three lines of `fermo_core.main.main_cli()`: load the JSON, validate it against the
schema, then let `ParameterManager` parse every block into typed, Pydantic-validated parameter objects
(file paths are checked for existence here too).

In [ ]:
start_time = datetime.now()

user_input = FileManager.load_json_file("example_data/case_study_parameters.json")
ValidationManager().validate_file_vs_jsonschema(user_input, "example_data/case_study_parameters.json")

param_manager = ParameterManager()
param_manager.assign_parameters_cli(user_input)

print("Parameters assigned. Output directory:", param_manager.OutputParameters.directory_path)

## 5. Stage 2 — parse the input files (`GeneralParser`)

`GeneralParser` reads the peak table, MS/MS spectra, phenotype/group metadata, spectral library, MS2Query
results and antiSMASH results, and builds three core objects:

- `stats`: a `Stats` object with run-wide summary information (feature/sample ID sets, RT/intensity ranges,
  networks, phenotype summaries, ...)
- `features`: a `Repository` mapping feature ID -> `Feature` object (one per molecular feature / MS peak)
- `samples`: a `Repository` mapping sample name -> `Sample` object

This is the step that actually reads and parses every file listed in the parameters — it can take a little
while on the full case study (antiSMASH parsing and MS2Query result matching in particular).

In [ ]:
general_parser = GeneralParser()
general_parser.parse_parameters(param_manager)
stats, features, samples = general_parser.return_attributes()

print(f"Parsed {len(features.entries)} molecular features across {len(samples.entries)} samples.")

## 6. Stage 3 — run the analysis modules (`AnalysisManager`)

`AnalysisManager.analyze()` runs every module enabled in the parameters file, in a fixed order: feature
filtering, blank assignment, sample-group assignment, group-factor assignment, phenotype-score assignment,
spectral-similarity networking (modified cosine + MS2DeepScore), annotation (adducts, neutral losses,
fragments, spectral library matching, antiSMASH knownclusterblast matching), score assignment, and
pseudo-chromatogram trace calculation. It mutates `stats` / `features` / `samples` in place and can also
update `params` (e.g. once dynamic values such as `filter_rel_int_range` are resolved against the loaded
data), so `return_attributes()` hands back all four.

In [ ]:
analysis_manager = AnalysisManager(
    params=param_manager, stats=stats, features=features, samples=samples
)
analysis_manager.analyze()
stats, features, samples, param_manager = analysis_manager.return_attributes()

print("Analysis complete.")

## 7. Stage 4 — export the results (`ExportManager`)

The final stage writes everything to `example_data/results/`:

- `out.fermo.session.json` — the full session, loadable directly into [FERMO Online](https://fermo.bioinformatics.nl/)
- `out.fermo.full.csv` / `out.fermo.abbrev.csv` — the annotated peak table as a spreadsheet
- `out.fermo.<network>.graphml` — Cytoscape-compatible spectral similarity networks (one per algorithm run)
- `out.fermo.summary.txt` — a human-readable summary of every module that ran and what it found

In [ ]:
export_manager = ExportManager(
    params=param_manager, stats=stats, features=features, samples=samples
)
export_manager.run(metadata.version("fermo_core"), start_time)

print("Results written to:", param_manager.OutputParameters.directory_path)
print(sorted(p.name for p in param_manager.OutputParameters.directory_path.iterdir()))

In [ ]:
with open(param_manager.OutputParameters.directory_path / "out.fermo.summary.txt") as f:
    print(f.read())

## 8. Recreate the README's result: feature 83, the siomycin hit

The FERMO README highlights that filtering by `Phenotype Score` on the dashboard surfaces a group of
features annotated as siomycins (thiopeptide antibiotics), with feature ID `83` as the example. We can
pull that feature straight out of the `features` repository without touching the GUI.

In [ ]:
feature_83 = features.get(83)

print(f"Feature ID:        {feature_83.f_id}")
print(f"m/z:                {feature_83.mz:.4f}")
print(f"Retention time:     {feature_83.rt:.2f} min")
print(f"Detected in samples:{sorted(feature_83.samples)}")
print(f"Phenotype score:    {feature_83.Scores.phenotype}")
print(f"Novelty score:      {feature_83.Scores.novelty}")

If spectral library or antiSMASH knownclusterblast matching found hits for this feature, they're on
`feature_83.Annotations.matches` (library hits) and networking results are under `feature_83.networks` —
this is where the "siomycin" annotation the README mentions would show up.

In [ ]:
if feature_83.Annotations and feature_83.Annotations.matches:
    for match in feature_83.Annotations.matches:
        print(f"{match.library:>20} | {match.algorithm:<12} | score={match.score:.2f} | id={match.id}")
else:
    print("No library/antiSMASH matches recorded for this feature "
          "(depends on which modules were enabled and their score cutoffs).")

## 9. Rank all features by phenotype score

The dashboard's `Phenotype Score` filter is just a sort/threshold over this same value across every
feature — reproducing it in plain pandas:

In [ ]:
import pandas as pd

rows = []
for f_id, feat in features.entries.items():
    rows.append({
        "f_id": f_id,
        "mz": feat.mz,
        "rt": feat.rt,
        "phenotype_score": feat.Scores.phenotype if feat.Scores else None,
        "novelty_score": feat.Scores.novelty if feat.Scores else None,
        "n_samples": len(feat.samples) if feat.samples else 0,
    })

df_features = pd.DataFrame(rows).sort_values("phenotype_score", ascending=False)
df_features.head(15)

## 10. Next steps

- Load `example_data/results/out.fermo.session.json` into [FERMO Online](https://fermo.bioinformatics.nl/)
  to browse the same results interactively, with the chromatogram traces, spectral similarity network, and
  filter panel from the dashboard.
- To run this on your own data, point every `*Parameters.filepath` in a copy of
  `case_study_parameters.json` at your files — accepted formats for each input type are listed in the
  [FERMO documentation](https://fermo-metabolomics.github.io/fermo_docs/home/input_output/).
- Optional modules (spectral library matching, antiSMASH matching, MS2DeepScore networking, quantitative
  phenotype correlation, ...) can be toggled independently via each block's `"activate_module"` flag —
  set a block to `false` or delete it entirely to skip that stage.